# Example 2 — Building a RAG System with LlamaIndex

## The Problem RAG Solves

Large language models like GPT-4 are extraordinarily capable — but their knowledge has a hard cutoff date, and they have *zero* awareness of your private documents. Ask ChatGPT about an internal company report, a paper published last week, or earnings released yesterday: it simply doesn't know.

**Retrieval-Augmented Generation (RAG)** bridges this gap. Instead of relying purely on memorised knowledge, a RAG system *retrieves* the most relevant passages from your own documents and injects them as context before the model generates an answer.

### The Open-Book Exam Analogy

> Imagine sitting an open-book exam. You don't need to memorise every fact — you know *how to find* the right page quickly. RAG works the same way: the AI looks up the answer in the provided documents *before* it writes a single word.

### How RAG Works — Three Steps

```
User Question
     │
     ▼
┌─────────────────────────────────────────────┐
│  Step 1 — EMBED                             │
│  Your question is converted to a numerical  │
│  vector ("embedding") by an embedding model │
└─────────────────────────────────────────────┘
     │
     ▼
┌─────────────────────────────────────────────┐
│  Step 2 — RETRIEVE                          │
│  The top-K most semantically similar        │
│  document chunks are fetched from the index │
└─────────────────────────────────────────────┘
     │
     ▼
┌─────────────────────────────────────────────┐
│  Step 3 — GENERATE                          │
│  Question + retrieved chunks are sent to    │
│  the LLM together → grounded answer         │
└─────────────────────────────────────────────┘
     │
     ▼
  Answer grounded in YOUR documents ✓
```

## What You'll Learn

- How to load real-world documents with **LlamaIndex's** `SimpleDirectoryReader`
- What *vector embeddings* are and why they enable semantic search
- How to build a queryable vector index from a PDF in minutes
- How to ask natural-language questions and get answers grounded in your data

## The Dataset: TCS Annual Report 2022–23

We'll use the **Tata Consultancy Services (TCS) Annual Report 2022–23** — a 200+ page document covering financial results, CEO commentary, strategic priorities, talent metrics, and ESG commitments. This is exactly the kind of dense document that normally requires hours to read; by the end of this notebook, you'll query it conversationally in seconds.

## Before You Start

- Confirm your `.env` file contains `OPENAI_API_KEY=sk-...`
- The file `annual-report-2022-2023.pdf` should already be in the `data/` folder
- Run all cells **top to bottom** — each cell depends on the one above

In [1]:
import os
from llama_index.core import SimpleDirectoryReader
from llama_index.core import GPTVectorStoreIndex  
from dotenv import load_dotenv

The cell above imports the two packages we need throughout this notebook:

- **`llama_index`** — handles document loading, chunking, embedding, and retrieval. What would take ~300 lines of custom code is reduced to a handful of function calls.
- **`python-dotenv`** — loads your OpenAI API key from a `.env` file so you never hard-code secrets in notebooks.

Now let's load the API key and move on to Step 2.

In [2]:
load_dotenv()

True

---

## Step 2 — Load Your Documents

`SimpleDirectoryReader` scans a folder and loads every supported file — PDFs, Word docs, plain text, Markdown, HTML — into a unified list of `Document` objects.

Each `Document` contains:
- **Raw text** — extracted from the file (LlamaIndex uses `pypdf` for PDFs, page by page)
- **Metadata** — file name, page number, creation date, etc.

These metadata fields become very useful later: when the model answers a question, it can cite *which page* of *which document* the answer came from.

> ⏱ **Expected output:** You should see the number of document chunks loaded printed below the cell.

In [3]:
documents = SimpleDirectoryReader("data").load_data()
print(f"✅ Loaded {len(documents)} document pages from the TCS Annual Report.")

✅ Loaded 346 document pages from the TCS Annual Report.


---

## Step 3 — Build the Vector Index

This is where the heavy lifting happens. `GPTVectorStoreIndex.from_documents()` performs three operations automatically:

1. **Chunk** — Splits each page into smaller overlapping text segments (~512 tokens each with 20-token overlap). Smaller chunks = more precise retrieval.
2. **Embed** — Sends each chunk to OpenAI's `text-embedding-ada-002` model, which converts the text into a **1,536-dimensional numerical vector**.
3. **Store** — Saves all vectors in an in-memory vector store, indexed for fast similarity search.

### What Exactly Is a Vector Embedding?

Think of it as assigning a *location in meaning-space* to a piece of text. Sentences with similar meanings end up **geometrically close** to each other — even if they use entirely different words. This enables true *semantic* search:

```
"The CEO earned $2M in performance bonuses"  →  [0.23, -0.41, 0.87, ...]
"Executive compensation included incentives"  →  [0.21, -0.39, 0.85, ...]  ← CLOSE ✓
"The weather in Mumbai was extremely hot"     →  [0.91,  0.12, -0.44, ...]  ← FAR AWAY ✓
```

A keyword search would miss the first match ("CEO" ≠ "Executive"). Embedding-based search catches it.

> ⚠️ **This cell makes API calls to OpenAI and takes 30–90 seconds.** You'll be charged a small fraction of a cent for the embedding calls.

In [4]:
if 'OPENAI_API_KEY' not in os.environ:
    os.environ['OPENAI_API_KEY'] = ''
    

In [5]:
print("⏳ Building vector index... (this may take 30–90 seconds)")
index = GPTVectorStoreIndex.from_documents(documents)
print("✅ Index built successfully! Ready to answer questions.")

⏳ Building vector index... (this may take 30–90 seconds)


2026-03-14 21:16:32,862 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-14 21:16:34,301 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-14 21:16:35,246 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-14 21:16:36,537 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-14 21:16:37,651 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


✅ Index built successfully! Ready to answer questions.


---

## Step 4 — Query Your Documents in Natural Language

`index.as_query_engine()` creates an interface that, for **every question you ask**, does the following automatically:

1. Embeds your question into a vector
2. Finds the **top-K most similar** document chunks (default K=2)
3. Constructs a prompt: *"Given the following context: [retrieved chunks]... please answer: [your question]"*
4. Sends the combined prompt to the LLM and streams back the response

The key insight: the LLM is **never guessing from memory**. Every answer is grounded in text that was actually retrieved from the annual report. This dramatically reduces hallucinations.

Let's try three progressively interesting questions:

In [6]:
query_engine = index.as_query_engine()

# --- Query 1: Leadership & Strategy ---
print("=" * 60)
print("QUERY 1: CEO Commentary & Strategic Priorities")
print("=" * 60)
response1 = query_engine.query(
    "Who was the CEO of TCS in 2023 and what were his key messages "
    "in the annual letter? What strategic priorities did he emphasise?"
)
print(response1)

QUERY 1: CEO Commentary & Strategic Priorities


2026-03-14 21:16:38,339 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-14 21:16:40,805 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The CEO of TCS in 2023 was K Krithivasan. In his annual letter, he expressed gratitude for the opportunity to lead the organization and highlighted the importance of customer relationships, technology adoption, and sustainability. He emphasized strategic priorities such as focusing on cloud adoption, data architecture, customer experience, business model transformation, and addressing net-zero carbon emission targets. Additionally, he mentioned the significance of technologies like 5G, IoT, generative AI, virtual reality/metaverse, and digital twin in driving business growth and transformation for clients.


In [7]:
# --- Query 2: Financial Performance ---
print("\n" + "=" * 60)
print("QUERY 2: Financial Performance")
print("=" * 60)
response2 = query_engine.query(
    "What were TCS's revenue and net profit figures for FY2023? "
    "How did they compare to the previous year, and which business "
    "segments or geographies drove the growth?"
)
print(response2)

# --- Query 3: People & Talent ---
print("\n" + "=" * 60)
print("QUERY 3: Workforce & Talent Strategy")
print("=" * 60)
response3 = query_engine.query(
    "How many employees does TCS have and what is their attrition rate? "
    "What initiatives has TCS launched to upskill its workforce in AI and cloud?"
)
print(response3)

# --- Query 4: ESG & Sustainability ---
print("\n" + "=" * 60)
print("QUERY 4: Sustainability Commitments")
print("=" * 60)
response4 = query_engine.query(
    "What specific commitments has TCS made around net-zero carbon emissions "
    "and sustainability? Include any targets, timelines, or measurable goals mentioned."
)
print(response4)


QUERY 2: Financial Performance


2026-03-14 21:16:41,080 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-14 21:16:43,265 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


TCS's revenue for FY2023 was ₹2,25,458 crore and the net profit attributable to shareholders of the company was ₹42,147 crore. In comparison to the previous year, revenue increased by 17.6% and net profit grew by 10.0%. The growth was primarily driven by business growth in segments such as Banking, Financial Services and Insurance, Communication, Media and Technology, as well as growth in geographies like North America and the United Kingdom.

QUERY 3: Workforce & Talent Strategy


2026-03-14 21:16:43,499 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-14 21:16:44,781 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


TCS has more than 320,000 employees. The attrition rate for TCS in the IT services sector was 20.1% on an LTM basis. TCS has launched initiatives like the Elevate program to empower employees to take control of their careers and pursue their aspirations by achieving certain learning goals, which can result in significant pay increases. Additionally, TCS has focused on leadership development programs like iExcel to drive change and promote gender diversity within the organization.

QUERY 4: Sustainability Commitments


2026-03-14 21:16:45,081 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-14 21:16:47,279 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


TCS has committed to achieving a 70% reduction of Scope 1 + 2 emissions by 2025 compared to the base year of 2016, with a further goal of reaching Net Zero emissions by 2030. The company aims to achieve these targets by prioritizing energy optimization, increasing the use of renewable energy sources, and implementing initiatives such as Clever Energy to optimize energy consumption. Additionally, TCS plans to leverage its Vision 25x25 strategy to reduce emissions related to employee commutes and business travel.


---

## Try It Yourself — Challenges

Now that the index is live, experiment on your own:

**Challenge 1 — Ask something granular**

Replace the query string with a highly specific question, e.g.:
> *"What is TCS's deal pipeline value for FY2023, and how does it compare to FY2022?"*

**Challenge 2 — Test the hallucination boundary**

Ask about something that is *not* in the report, e.g.:
> *"What did TCS announce at their Q1 FY2025 earnings call?"*

Observe: does the model hallucinate a confident answer, or does it correctly say it doesn't have that information? This tells you a lot about where RAG still has failure modes.

**Challenge 3 — Tune retrieval depth**

```python
# Default retrieves top-2 chunks. Try increasing it:
query_engine = index.as_query_engine(similarity_top_k=5)
```

Does retrieving more context improve answer quality? Does it ever introduce noise or contradictions?

**Challenge 4 — Add your own document**

Drop any PDF (a research paper, a company report, your CV) into the `data/` folder, re-run Steps 2–4, and start querying. This is RAG applied to *your* data.

---

## Summary

You just built a complete RAG pipeline from scratch using four lines of meaningful code:

| Step | What You Did | LlamaIndex Component |
|------|-------------|---------------------|
| **Load** | Read a 200+ page PDF from disk | `SimpleDirectoryReader` |
| **Index** | Chunk → embed → store as vectors | `GPTVectorStoreIndex` |
| **Query** | Retrieve relevant chunks → synthesise answer | `QueryEngine` |

### Key Takeaways

**RAG ≠ fine-tuning.** RAG doesn't touch the model's weights. It provides fresh context at query time — making it fast, cheap to update, and auditable. Fine-tuning teaches the model new *behaviour*; RAG teaches it new *facts*.

**Embeddings = semantic GPS.** The quality of your retrieval depends on how well the embedding model captures meaning. OpenAI's `ada-002` is a strong general-purpose choice, but domain-specific embedding models can outperform it on niche corpora.

**Chunk size is a real design decision.** Too small → insufficient context per chunk. Too large → noisy, irrelevant text dilutes the prompt. LlamaIndex's defaults are sensible starting points, but tuning them is often where real-world RAG performance is won or lost.

### What's Next?

In the next notebook we'll use **LangGraph** to build a *multi-agent workflow* — where specialised agents work together, each reading and writing to a shared state, to accomplish tasks that a single LLM call could never handle alone.